# Pipeline

In [6]:
!pip install -q mlflow dagshub lightgbm catboost

In [12]:
import numpy as np
import pandas as pd
import dagshub
import mlflow
import mlflow.sklearn

from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline

from catboost import CatBoostClassifier

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None


dagshub.init(repo_owner='Sula1909', repo_name='ML-Assignment2', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/Sula1909/ML-Assignment2.mlflow')
mlflow.set_experiment('CatBoost_Training')


class ColumnRenameTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.rename_map_ = {c: c.replace('-', '_') for c in X.columns if '-' in c}
        return self

    def transform(self, X):
        X_out = X.copy()
        if self.rename_map_:
            X_out = X_out.rename(columns=self.rename_map_)
        return X_out


def reduce_mem_usage(df):
    df = df.copy()
    for col in df.columns:
        col_type = df[col].dtype

        if pd.api.types.is_datetime64_any_dtype(col_type):
            continue

        if pd.api.types.is_numeric_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()

            if pd.api.types.is_integer_dtype(col_type):
                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    return df


class MemoryReductionTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return reduce_mem_usage(X)


class OutlierRemovalTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, amt_col='TransactionAmt', threshold=30000, drop_on_inference=False):
        self.amt_col = amt_col
        self.threshold = threshold
        self.drop_on_inference = drop_on_inference

    def fit(self, X, y=None):
        return self

    def fit_transform(self, X, y=None, **fit_params):
        self.fit(X, y)
        X_out = X.copy()
        if self.amt_col in X_out.columns:
            mask = X_out[self.amt_col].fillna(-np.inf) <= self.threshold
            X_out = X_out.loc[mask].copy()
        return X_out

    def transform(self, X):
        X_out = X.copy()
        if self.drop_on_inference and self.amt_col in X_out.columns:
            mask = X_out[self.amt_col].fillna(-np.inf) <= self.threshold
            X_out = X_out.loc[mask].copy()
        return X_out


class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, transactiondt_col='TransactionDT', amt_col='TransactionAmt', card1_col='card1'):
        self.transactiondt_col = transactiondt_col
        self.amt_col = amt_col
        self.card1_col = card1_col

    def fit(self, X, y=None):
        if self.card1_col in X.columns and self.amt_col in X.columns:
            self.card1_amt_mean_ = X.groupby(self.card1_col)[self.amt_col].mean()
        else:
            self.card1_amt_mean_ = pd.Series(dtype=np.float64)
        return self

    def transform(self, X):
        X_out = X.copy()

        if self.transactiondt_col in X_out.columns:
            t = X_out[self.transactiondt_col].fillna(0)
            X_out['day'] = (t // (24 * 60 * 60)).astype(np.int32)
            X_out['hour'] = ((t // 3600) % 24).astype(np.int8)

            def hour_feature(hour):
                if hour > 4 and hour < 12:
                    return 'highalert'
                if hour == 12 or hour == 19:
                    return 'lowalert'
                if hour == 3 or hour == 4 or hour == 24:
                    return 'mediumalert'
                return 'noalert'

            X_out['hour_alertFeature'] = X_out['hour'].apply(hour_feature)
        else:
            X_out['day'] = 0
            X_out['hour'] = 0
            X_out['hour_alertFeature'] = 'noalert'

        if self.card1_col in X_out.columns and self.amt_col in X_out.columns and len(self.card1_amt_mean_) > 0:
            mean_amt = X_out[self.card1_col].map(self.card1_amt_mean_)
            ratio = X_out[self.amt_col] / mean_amt
            X_out['TransactionAmt_to_mean_card1'] = ratio.replace([np.inf, -np.inf], np.nan)
        else:
            X_out['TransactionAmt_to_mean_card1'] = np.nan

        return X_out


class CorrelationFilterTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, prefix, threshold=0.95):
        self.prefix = prefix
        self.threshold = threshold

    def fit(self, X, y=None):
        target_cols = [c for c in X.columns if c.startswith(self.prefix)]
        target_cols = [c for c in target_cols if pd.api.types.is_numeric_dtype(X[c])]

        self.drop_cols_ = []
        if len(target_cols) > 1:
            corr = X[target_cols].corr().abs()
            upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
            self.drop_cols_ = [col for col in upper.columns if any(upper[col] > self.threshold)]
        return self

    def transform(self, X):
        X_out = X.copy()
        to_drop = [c for c in self.drop_cols_ if c in X_out.columns]
        if to_drop:
            X_out = X_out.drop(columns=to_drop)
        return X_out


class LGBMFeatureSelectionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, prefix='V', importance_threshold=3, random_state=42):
        self.prefix = prefix
        self.importance_threshold = importance_threshold
        self.random_state = random_state

    def fit(self, X, y=None):
        self.drop_cols_ = []

        if y is None:
            return self

        v_cols = [c for c in X.columns if c.startswith(self.prefix)]
        v_cols = [c for c in v_cols if pd.api.types.is_numeric_dtype(X[c])]

        if len(v_cols) == 0:
            return self

        if LGBMClassifier is None:
            return self

        model = LGBMClassifier(
            n_estimators=200,
            learning_rate=0.05,
            random_state=self.random_state,
            n_jobs=-1
        )

        y_fit = y.loc[X.index] if isinstance(y, pd.Series) else y
        X_fit = X[v_cols].fillna(-999)
        model.fit(X_fit, y_fit)

        importances = pd.Series(model.feature_importances_, index=v_cols)
        self.drop_cols_ = importances[importances < self.importance_threshold].index.tolist()
        return self

    def transform(self, X):
        X_out = X.copy()
        to_drop = [c for c in self.drop_cols_ if c in X_out.columns]
        if to_drop:
            X_out = X_out.drop(columns=to_drop)
        return X_out


class FrequencyEncodingTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        self.freq_maps_ = {}
        for col in self.columns:
            if col in X.columns:
                self.freq_maps_[col] = X[col].value_counts(dropna=False, normalize=True)
        return self

    def transform(self, X):
        X_out = X.copy()
        for col, mapping in self.freq_maps_.items():
            X_out[f'{col}_freq_enc'] = X_out[col].map(mapping).astype(np.float32)
        return X_out


class WOEEncodingTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns, event=1, smoothing=0.5):
        self.columns = columns
        self.event = event
        self.smoothing = smoothing

    def fit(self, X, y=None):
        if y is None:
            raise ValueError('WOEEncodingTransformer requires y in fit().')

        y_series = y.loc[X.index] if isinstance(y, pd.Series) else pd.Series(y, index=X.index)

        event_total = (y_series == self.event).sum()
        non_event_total = (y_series != self.event).sum()

        self.woe_maps_ = {}
        self.default_woe_ = {}

        for col in self.columns:
            if col not in X.columns:
                continue

            tmp = pd.DataFrame({
                'x': X[col].astype(str).fillna('__MISSING__'),
                'y': y_series
            })

            grouped = tmp.groupby('x')['y'].agg(['count', 'sum'])
            grouped['event_count'] = grouped['sum']
            grouped['non_event_count'] = grouped['count'] - grouped['sum']

            grouped['event_dist'] = (grouped['event_count'] + self.smoothing) / (event_total + self.smoothing * len(grouped))
            grouped['non_event_dist'] = (grouped['non_event_count'] + self.smoothing) / (non_event_total + self.smoothing * len(grouped))
            grouped['woe'] = np.log(grouped['event_dist'] / grouped['non_event_dist'])

            self.woe_maps_[col] = grouped['woe'].to_dict()
            self.default_woe_[col] = float(grouped['woe'].mean())

        return self

    def transform(self, X):
        X_out = X.copy()
        for col, mapping in self.woe_maps_.items():
            as_str = X_out[col].astype(str).fillna('__MISSING__')
            X_out[col] = as_str.map(mapping).fillna(self.default_woe_[col]).astype(np.float32)
        return X_out


class LowCardinalityOHETransformer(BaseEstimator, TransformerMixin):
    def __init__(self, max_unique=5):
        self.max_unique = max_unique

    def fit(self, X, y=None):
        cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
        self.ohe_cols_ = [c for c in cat_cols if X[c].nunique(dropna=False) <= self.max_unique]

        dummies = pd.get_dummies(
            X[self.ohe_cols_],
            columns=self.ohe_cols_,
            dummy_na=True,
            dtype=np.uint8
        ) if self.ohe_cols_ else pd.DataFrame(index=X.index)

        self.dummy_cols_ = dummies.columns.tolist()
        return self

    def transform(self, X):
        X_out = X.copy()

        if self.ohe_cols_:
            dummies = pd.get_dummies(
                X_out[self.ohe_cols_],
                columns=self.ohe_cols_,
                dummy_na=True,
                dtype=np.uint8
            )
            dummies = dummies.reindex(columns=self.dummy_cols_, fill_value=0)
            X_out = X_out.drop(columns=[c for c in self.ohe_cols_ if c in X_out.columns])
            X_out = pd.concat([X_out, dummies], axis=1)

        return X_out


class HighCardinalityLabelEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, min_unique=6):
        self.min_unique = min_unique

    def fit(self, X, y=None):
        cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
        self.label_cols_ = [c for c in cat_cols if X[c].nunique(dropna=False) >= self.min_unique]
        self.label_maps_ = {}

        for col in self.label_cols_:
            values = pd.Series(X[col].astype(str).fillna('__MISSING__').unique())
            self.label_maps_[col] = {v: i for i, v in enumerate(values)}

        return self

    def transform(self, X):
        X_out = X.copy()

        for col in self.label_cols_:
            if col in X_out.columns:
                as_str = X_out[col].astype(str).fillna('__MISSING__')
                X_out[col] = as_str.map(self.label_maps_[col]).fillna(-1).astype(np.int32)

        return X_out


class MedianImputerTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.medians_ = X.median(numeric_only=True)
        return self

    def transform(self, X):
        X_out = X.copy()
        num_cols = X_out.select_dtypes(include=[np.number]).columns
        X_out[num_cols] = X_out[num_cols].fillna(self.medians_)
        return X_out


class IndexAlignedCatBoostClassifier(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        iterations=700,
        learning_rate=0.05,
        depth=7,
        l2_leaf_reg=4,
        loss_function='Logloss',
        eval_metric='AUC',
        bootstrap_type='Bernoulli',
        subsample=0.9,
        rsm=0.8,
        random_strength=1.0,
        leaf_estimation_iterations=10,
        auto_class_weights='Balanced',
        random_seed=42,
        verbose=False
    ):
        self.iterations = iterations
        self.learning_rate = learning_rate
        self.depth = depth
        self.l2_leaf_reg = l2_leaf_reg
        self.loss_function = loss_function
        self.eval_metric = eval_metric
        self.bootstrap_type = bootstrap_type
        self.subsample = subsample
        self.rsm = rsm
        self.random_strength = random_strength
        self.leaf_estimation_iterations = leaf_estimation_iterations
        self.auto_class_weights = auto_class_weights
        self.random_seed = random_seed
        self.verbose = verbose

    def _build_model(self):
        return CatBoostClassifier(
            iterations=self.iterations,
            learning_rate=self.learning_rate,
            depth=self.depth,
            l2_leaf_reg=self.l2_leaf_reg,
            loss_function=self.loss_function,
            eval_metric=self.eval_metric,
            bootstrap_type=self.bootstrap_type,
            subsample=self.subsample,
            rsm=self.rsm,
            random_strength=self.random_strength,
            leaf_estimation_iterations=self.leaf_estimation_iterations,
            auto_class_weights=self.auto_class_weights,
            random_seed=self.random_seed,
            verbose=self.verbose
        )

    def fit(self, X, y):
        self.model_ = self._build_model()

        if isinstance(y, pd.Series):
            y_fit = y.loc[X.index] if len(y) != len(X) else y
        else:
            y_fit = y

        self.model_.fit(X, y_fit)
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)


freq_cols = [
    'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2',
    'P_emaildomain', 'R_emaildomain'
]

woe_cols = [
    'P_emaildomain', 'R_emaildomain', 'id_30', 'id_31', 'id_33', 'DeviceInfo'
]

fraud_pipeline = Pipeline(steps=[
    ('rename_cols', ColumnRenameTransformer()),
    ('reduce_mem', MemoryReductionTransformer()),
    ('remove_outliers', OutlierRemovalTransformer(amt_col='TransactionAmt', threshold=30000)),
    ('feature_engineering', FeatureEngineeringTransformer()),
    ('corr_filter_c', CorrelationFilterTransformer(prefix='C', threshold=0.95)),
    ('corr_filter_d', CorrelationFilterTransformer(prefix='D', threshold=0.95)),
    ('lgbm_v_selector', LGBMFeatureSelectionTransformer(prefix='V', importance_threshold=3, random_state=42)),
    ('freq_encoding', FrequencyEncodingTransformer(columns=freq_cols)),
    ('woe_encoding', WOEEncodingTransformer(columns=woe_cols, event=1, smoothing=0.5)),
    ('ohe_low_cardinality', LowCardinalityOHETransformer(max_unique=5)),
    ('label_encode_high_cardinality', HighCardinalityLabelEncoderTransformer(min_unique=6)),
    ('median_impute', MedianImputerTransformer()),
    ('catboost_model', IndexAlignedCatBoostClassifier(
        iterations=700,
        learning_rate=0.05,
        depth=7,
        l2_leaf_reg=4,
        loss_function='Logloss',
        eval_metric='AUC',
        bootstrap_type='Bernoulli',
        subsample=0.9,
        rsm=0.8,
        random_strength=1.0,
        leaf_estimation_iterations=10,
        auto_class_weights='Balanced',
        random_seed=42,
        verbose=False
    ))
])


# Raw merged dataframe (no manual preprocessing)
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
raw_train_merged = train_transaction.merge(train_identity, how='left', on='TransactionID')

X = raw_train_merged.drop(columns=['isFraud', 'TransactionID'])
y = raw_train_merged['isFraud']

if 'TransactionDT' not in X.columns:
    raise ValueError('TransactionDT is required for chronological split.')

sorted_idx = X.sort_values('TransactionDT').index
n_rows = len(sorted_idx)
train_end = int(n_rows * 0.70)
valid_end = int(n_rows * 0.85)

train_idx = sorted_idx[:train_end]
valid_idx = sorted_idx[train_end:valid_end]
test_idx = sorted_idx[valid_end:]

X_train, y_train = X.loc[train_idx], y.loc[train_idx]
X_valid, y_valid = X.loc[valid_idx], y.loc[valid_idx]
X_test, y_test = X.loc[test_idx], y.loc[test_idx]

fraud_pipeline.fit(X_train, y_train)
train_pred = fraud_pipeline.predict_proba(X_train)[:, 1]
valid_pred = fraud_pipeline.predict_proba(X_valid)[:, 1]
test_pred = fraud_pipeline.predict_proba(X_test)[:, 1]

valid_label_pred = (valid_pred >= 0.50).astype(int)
test_label_pred = (test_pred >= 0.50).astype(int)

train_auc = roc_auc_score(y_train, train_pred)
valid_auc = roc_auc_score(y_valid, valid_pred)
test_auc = roc_auc_score(y_test, test_pred)

valid_precision = precision_score(y_valid, valid_label_pred, zero_division=0)
valid_recall = recall_score(y_valid, valid_label_pred, zero_division=0)
valid_f1 = f1_score(y_valid, valid_label_pred, zero_division=0)

test_precision = precision_score(y_test, test_label_pred, zero_division=0)
test_recall = recall_score(y_test, test_label_pred, zero_division=0)
test_f1 = f1_score(y_test, test_label_pred, zero_division=0)

print(" ")
print(f'Train AUC: {train_auc:.6f}')
print(f'Validation AUC: {valid_auc:.6f}')
print(f'Test AUC: {test_auc:.6f}')
print(' ')
print(f'Validation -> Precision: {valid_precision:.6f} | Recall: {valid_recall:.6f} | F1: {valid_f1:.6f}')
print(f'Test -> Precision: {test_precision:.6f} | Recall: {test_recall:.6f} | F1: {test_f1:.6f}')


mlflow_params = {
    'iterations': 700,
    'learning_rate': 0.05,
    'depth': 7,
    'l2_leaf_reg': 4,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'bootstrap_type': 'Bernoulli',
    'subsample': 0.9,
    'rsm': 0.8,
    'random_strength': 1.0,
    'leaf_estimation_iterations': 10,
    'auto_class_weights': 'Balanced',
    'random_seed': 42,
    'classification_threshold': 0.50,
}

with mlflow.start_run(run_name='CatBoost_Pipeline'):
    mlflow.log_params(mlflow_params)

    mlflow.log_metric('Train_AUC', float(train_auc))
    mlflow.log_metric('Validation_AUC', float(valid_auc))
    mlflow.log_metric('Test_AUC', float(test_auc))

    mlflow.log_metric('Precision', float(valid_precision))
    mlflow.log_metric('Recall', float(valid_recall))
    mlflow.log_metric('F1', float(valid_f1))

    mlflow.log_metric('Test_Precision', float(test_precision))
    mlflow.log_metric('Test_Recall', float(test_recall))
    mlflow.log_metric('Test_F1', float(test_f1))

    mlflow.sklearn.log_model(sk_model=fraud_pipeline, artifact_path='model')

Initialized MLflow to track repo "Sula1909/ML-Assignment2"

Repository Sula1909/ML-Assignment2 initialized!

[LightGBM] [Info] Number of positive: 14538, number of negative: 398838
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.703295 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23373
[LightGBM] [Info] Number of data points in the train set: 413376, number of used features: 337
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035169 -> initscore=-3.311789
[LightGBM] [Info] Start training from score -3.311789
 
Train AUC: 0.974251
Validation AUC: 0.916905
Test AUC: 0.904117
 
Validation -> Precision: 0.264943 | Recall: 0.690664 | F1: 0.382975
Test -> Precision: 0.249557 | Recall: 0.685371 | F1: 0.365887


2026/05/04 18:25:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 18:25:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run CatBoost_Pipeline at: https://dagshub.com/Sula1909/ML-Assignment2.mlflow/#/experiments/3/runs/84207cf7ce8a4ca087a6c358f0d9b488
🧪 View experiment at: https://dagshub.com/Sula1909/ML-Assignment2.mlflow/#/experiments/3
